**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---

## Qué son las salidas estructuradas

Un agente creado con `create_agent` puede devolver la respuesta final en **texto libre** (el contenido del último mensaje del asistente) o en un **formato fijo** que tu aplicación pueda consumir directamente. Esta segunda opción se conoce como **salida estructurada** (structured output): en lugar de parsear lenguaje natural, obtienes datos en forma de objeto Pydantic, diccionario tipado o JSON validado.

LangChain gestiona la salida estructurada mediante el parámetro **`response_format`** de `create_agent`. El usuario define un esquema (por ejemplo una clase Pydantic) y, cuando el modelo genera los datos acordes a ese esquema, LangChain los captura, valida y los expone en la clave **`'structured_response'`** del estado final del agente.

> Usar `response_format` permite integrar el agente en pipelines: la respuesta queda en un formato predecible (por ejemplo un DTO o un contrato de API) sin tener que escribir parsers propios.

## El parámetro response_format

En la firma de `create_agent`, `response_format` acepta:

* **`None`**: no se pide salida estructurada; la respuesta es solo el contenido del último mensaje.
* **Un tipo de esquema** (por ejemplo una clase Pydantic): LangChain elige automáticamente la estrategia según las capacidades del modelo.
* **`ProviderStrategy(schema)`**: usa la generación de salida estructurada nativa del proveedor (OpenAI, Anthropic, etc.) cuando está disponible.
* **`ToolStrategy(schema)`**: simula una herramienta para que el modelo “llame” con los datos estructurados; funciona con cualquier modelo que soporte tool calling.

Si pasas **solo el tipo** (por ejemplo `response_format=MiClasePydantic`), el comportamiento es:

* Si el modelo y el proveedor soportan **structured output nativo**, se usa **ProviderStrategy**.
* En caso contrario se usa **ToolStrategy**.

Ambas estrategias rellenan **`result["structured_response"]`** con la instancia validada (Pydantic) o el diccionario correspondiente.

```mermaid
flowchart LR
    subgraph input
        U[Usuario]
    end
    subgraph agent
        A[create_agent]
        T[Tools]
        R[response_format]
    end
    subgraph output
        MSG[messages]
        SR["structured_response"]
    end
    U --> A
    A --> T
    A --> R
    A --> MSG
    A --> SR
```

## Definir el esquema con Pydantic

El esquema de la respuesta se define con un modelo **Pydantic**: campos tipados y descripciones que guían al modelo. Las descripciones con `Field(description=...)` mejoran la calidad de lo que genera el LLM.

Ejemplo de esquema para un resumen de búsqueda que el agente rellenará después de usar herramientas:

In [1]:
from pydantic import BaseModel, Field

class ResumenBusqueda(BaseModel):
    """Resumen estructurado de una búsqueda en web."""
    consulta: str = Field(description="Consulta original del usuario")
    resumen: str = Field(description="Resumen en 2-3 frases de los resultados")
    fuentes: list[str] = Field(description="URLs de las fuentes utilizadas, como lista")
    respuesta_corta: str = Field(description="Una frase directa para el usuario")

Al pasar `ResumenBusqueda` como `response_format`, el agente debe devolver un objeto que cumpla ese esquema; LangChain lo valida y lo deja en `result["structured_response"]`.

## Agente con herramientas y salida estructurada

Un caso típico es un agente que **usa una o varias herramientas** (por ejemplo búsqueda web o APIs) y debe responder siempre con un **objeto Pydantic** en lugar de texto libre. El modelo puede hacer varias llamadas a herramientas y, al final, emitir la respuesta en el formato indicado.

Requisito importante: el modelo debe soportar **uso simultáneo de herramientas y salida estructurada**. Los modelos recientes de OpenAI, Anthropic y otros suelen soportarlo.

Ejemplo: agente con búsqueda web que devuelve un `ResumenBusqueda`:

In [2]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field

class ResumenBusqueda(BaseModel):
    """Resumen estructurado de una búsqueda en web."""
    consulta: str = Field(description="Consulta original del usuario")
    resumen: str = Field(description="Resumen en 2-3 frases de los resultados")
    fuentes: list[str] = Field(description="URLs de las fuentes utilizadas")
    respuesta_corta: str = Field(description="Una frase directa para el usuario")

web_search = TavilySearch(max_results=3)

agent = create_agent(
    model="gpt-5-nano",  # o el modelo que uses
    tools=[web_search],
    response_format=ResumenBusqueda,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Busca noticias recientes sobre lanzamientos de IA y dame un resumen."}]
})

# Respuesta estructurada lista para usar en código
datos = result["structured_response"]
print(datos.consulta)
print(datos.respuesta_corta)
print(datos.fuentes)

lanzamientos de IA noticias recientes
En la última semana destacaron lanzamientos de IA como HyperNova 60B comprimido de Multiverse, la fábrica inteligente de Machina Labs y la compra de Canopus AI por Siemens, junto con avances regulatorios en IA.
['https://techcrunch.com/2026/02/24/spanish-soonicorn-multiverse-computing-releases-free-compressed-ai-model/', 'https://www.ien.com/artificial-intelligence/news/22961006/ransomware-report-finds-ai-accelerating-not-replacing-humanled-attacks', 'https://www.govtech.com/artificial-intelligence/state-politics-affecting-trumps-push-to-limit-ai-regulation']


El agente puede invocar `tavily_search` varias veces si lo necesita; la **respuesta final** debe ajustarse al esquema `ResumenBusqueda` y es la que aparece en **`result["structured_response"]`**.

> La clave `structured_response` solo existe en el estado cuando has definido `response_format`. Si no la usas, debes leer la respuesta desde el último mensaje del asistente en `result["messages"]`.

## Estrategias: ProviderStrategy y ToolStrategy

Cuando quieres control explícito sobre la estrategia en lugar de la elección automática:

* **ProviderStrategy**: usa la API nativa de salida estructurada del proveedor. Es la más fiable cuando el modelo lo soporta (por ejemplo OpenAI con `response_format` en la API). Se importa desde `langchain.agents.structured_output`:

In [3]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    response_format=ProviderStrategy(ResumenBusqueda),
)

* **ToolStrategy**: el modelo “llama” a una herramienta ficticia cuyo esquema es tu Pydantic. Funciona con cualquier modelo que soporte tool calling y es el recurso cuando el proveedor no ofrece structured output nativo:

In [4]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    response_format=ToolStrategy(ResumenBusqueda),
)

Tiene sentido usar **ToolStrategy de forma explícita** cuando el modelo es antiguo (legacy), muy pequeño o es open source y **no está entrenado para structured outputs nativos** (p. ej. muchos modelos locales vía Ollama o proveedores sin API de respuesta estructurada). En esos casos el proveedor no puede devolver JSON conforme a esquema de forma nativa, pero sí puede invocar herramientas; ToolStrategy aprovecha esa capacidad para obtener la misma forma de salida. Si pasas solo el esquema, LangChain ya hace este fallback por ti cuando detecta que el modelo no soporta structured output nativo.

En ambos casos el resultado validado está en **`result["structured_response"]`**. Con **solo el esquema** (`response_format=ResumenBusqueda`) suele bastar; LangChain elige ProviderStrategy o ToolStrategy según el modelo.

## Ejemplo completo: agente con dos herramientas y Pydantic

El siguiente ejemplo define un agente con **dos herramientas** (coordenadas y tiempo) y una **salida estructurada** en Pydantic que resume lugar y temperatura. Así se ve en la práctica la combinación de varias herramientas y un único objeto de respuesta.

In [5]:
import requests
from langchain.tools import tool
from langchain.agents import create_agent
from pydantic import BaseModel, Field

@tool("get_coordinates")
def get_coordinates(location: str) -> str:
    """Obtener latitud y longitud a partir del nombre del lugar."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": location, "format": "json", "limit": 1}
    headers = {"User-Agent": "AgentDemo/1.0"}
    resp = requests.get(url, params=params, headers=headers)
    data = resp.json()
    if data:
        return f"{data[0]['lat']}, {data[0]['lon']}"
    return ""

@tool("get_weather")
def get_weather(latitude: str, longitude: str) -> str:
    """Obtener temperatura actual (C) para unas coordenadas."""
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    resp = requests.get(url)
    data = resp.json()
    return str(data.get("current", {}).get("temperature_2m", ""))

class InformeTiempo(BaseModel):
    """Informe de tiempo para un lugar."""
    lugar: str = Field(description="Nombre del lugar consultado")
    coordenadas: str = Field(description="Latitud y longitud en formato 'lat, lon'")
    temperatura_c: float = Field(description="Temperatura en grados Celsius")
    resumen: str = Field(description="Una frase resumiendo el tiempo en ese lugar")

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_coordinates, get_weather],
    response_format=InformeTiempo,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "¿Qué temperatura hace ahora en el Parque del Retiro, Madrid?"}]
})

informe = result["structured_response"]
print(informe)
# InformeTiempo(lugar='Parque del Retiro, Madrid', coordenadas='40.41, -3.68', temperatura_c=9.6, resumen='...')

print(informe.lugar)
print(f"{informe.temperatura_c} °C")
print(informe.resumen)

lugar='Parque del Retiro, Madrid' coordenadas='40.4149460, -3.6832845' temperatura_c=20.3 resumen='La temperatura actual en el Parque del Retiro es de 20.3 °C.'
Parque del Retiro, Madrid
20.3 °C
La temperatura actual en el Parque del Retiro es de 20.3 °C.


El agente decide cuándo llamar a `get_coordinates` y `get_weather`; al terminar, debe rellenar **`InformeTiempo`** con los datos obtenidos. Esa instancia es la que se devuelve en **`structured_response`**: puedes **imprimirla**, acceder a **`informe.lugar`**, **`informe.temperatura_c`**, etc., y usarla en lógica posterior (APIs, bases de datos, serialización).

Puedes ampliar el esquema (por ejemplo añadiendo humedad o sensación térmica) o cambiar las herramientas; el patrón es el mismo: **`create_agent`** con **`tools`** y **`response_format`** igual a tu clase Pydantic.